In [4]:
import re
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report
)

RANDOM_STATE = 42

df = pd.read_csv("final_resume_dataset.csv")

print("Original dataset:", df.shape)
print(df.columns.tolist())

Original dataset: (54933, 11)
['person_id', 'name', 'email', 'phone', 'linkedin', 'skill', 'program', 'title', 'firm', 'resume_text', 'ability']


In [5]:
df = df[
    [
        "skill",
        "program",
        "title",
        "firm",
        "resume_text",
        "ability"
    ]
].copy()

In [6]:
text_columns = [
    "skill",
    "program",
    "title",
    "firm",
    "resume_text",
    "ability"
]

for col in text_columns:
    df[col] = (
        df[col]
        .fillna("")
        .astype(str)
    )

In [7]:
def clean_text(text):

    text = str(text).lower()

    # Replace separators
    text = re.sub(r"[/|\\\-]", " ", text)

    # Remove emails
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Remove URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    # Keep letters and numbers
    text = re.sub(
        r"[^a-z0-9+#.\s]",
        " ",
        text
    )

    # Normalize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [8]:
for col in text_columns:
    df[col] = df[col].apply(clean_text)

In [9]:
def extract_role(title):

    t = clean_text(title)

    # Most specific roles first
    if "machine learning engineer" in t:
        return "Machine Learning Engineer"

    if "data scientist" in t:
        return "Data Scientist"

    if "data analyst" in t:
        return "Data Analyst"

    if "business analyst" in t:
        return "Business Analyst"

    if "backend developer" in t or "back end developer" in t:
        return "Backend Developer"

    if "full stack developer" in t or "fullstack developer" in t:
        return "Full Stack Developer"

    if "java developer" in t:
        return "Java Developer"

    if "python developer" in t:
        return "Python Developer"

    if "front end developer" in t or "frontend developer" in t:
        return "Front End Developer"

    if "web developer" in t:
        return "Front End Developer"

    if "database administrator" in t:
        return "Database Administrator"

    if "oracle database administrator" in t:
        return "Database Administrator"

    if "sql database administrator" in t:
        return "Database Administrator"

    if "network administrator" in t:
        return "Network Administrator"

    if "network engineer" in t:
        return "Network Administrator"

    if "it security analyst" in t:
        return "IT Security Analyst"

    if "security analyst" in t:
        return "IT Security Analyst"

    if "it project manager" in t:
        return "IT Project Manager"

    if "project manager" in t:
        return "Project Manager"

    if "software engineer" in t:
        return "Software Engineer"

    if "devops engineer" in t:
        return "DevOps Engineer"

    return None

In [10]:
df["career"] = df["title"].apply(
    extract_role
)

print(
    df["career"].value_counts()
)

career
Front End Developer          8145
Network Administrator        8025
Database Administrator       7632
Java Developer               5496
IT Security Analyst          5154
Project Manager              3675
IT Project Manager           3390
Python Developer             2091
Business Analyst             1833
Full Stack Developer         1539
Data Analyst                  861
Software Engineer             321
Backend Developer             210
Data Scientist                126
DevOps Engineer                24
Machine Learning Engineer      15
Name: count, dtype: int64


In [11]:
df = df.dropna(
    subset=["career"]
).reset_index(drop=True)

print(
    "After career extraction:",
    df.shape
)

After career extraction: (48537, 7)


In [12]:
df = df.drop_duplicates(
    subset=["resume_text"]
).reset_index(drop=True)

print(
    "After duplicate removal:",
    df.shape
)

After duplicate removal: (16152, 7)


In [13]:
df["resume_length"] = (
    df["resume_text"]
    .str.split()
    .str.len()
)

df = df[
    df["resume_length"] >= 20
].copy()

df = df.drop(
    columns=["resume_length"]
)

df = df.reset_index(
    drop=True
)

print(
    "After short-resume removal:",
    df.shape
)

After short-resume removal: (16065, 7)


In [14]:
df["profile_text"] = (
    "skills " +
    df["skill"] +
    " education " +
    df["program"] +
    " experience " +
    df["ability"] +
    " company " +
    df["firm"] +
    " resume " +
    df["resume_text"]
)

In [15]:
df["profile_text"] = df[
    "profile_text"
].apply(clean_text)

In [16]:
print(
    df["career"].value_counts()
)

career
Front End Developer          2700
Network Administrator        2637
Database Administrator       2528
Java Developer               1820
IT Security Analyst          1714
Project Manager              1218
IT Project Manager           1119
Python Developer              689
Business Analyst              610
Full Stack Developer          512
Data Analyst                  287
Software Engineer             107
Backend Developer              69
Data Scientist                 42
DevOps Engineer                 8
Machine Learning Engineer       5
Name: count, dtype: int64


In [17]:
MIN_CLASS_SIZE = 50

class_counts = df["career"].value_counts()

valid_classes = class_counts[
    class_counts >= MIN_CLASS_SIZE
].index

df = df[
    df["career"].isin(valid_classes)
].reset_index(drop=True)

print(
    "\nFINAL CLASS DISTRIBUTION"
)

print(
    df["career"].value_counts()
)


FINAL CLASS DISTRIBUTION
career
Front End Developer       2700
Network Administrator     2637
Database Administrator    2528
Java Developer            1820
IT Security Analyst       1714
Project Manager           1218
IT Project Manager        1119
Python Developer           689
Business Analyst           610
Full Stack Developer       512
Data Analyst               287
Software Engineer          107
Backend Developer           69
Name: count, dtype: int64


In [18]:
print("\n==============================")
print("FINAL PREPROCESSED DATASET")
print("==============================")

print(
    "Rows:",
    len(df)
)

print(
    "Classes:",
    df["career"].nunique()
)

print(
    "\nClass distribution:"
)

print(
    df["career"].value_counts()
)

print(
    "\nSample profile:"
)

print(
    df[
        [
            "career",
            "profile_text"
        ]
    ].head(2)
)


FINAL PREPROCESSED DATASET
Rows: 16010
Classes: 13

Class distribution:
career
Front End Developer       2700
Network Administrator     2637
Database Administrator    2528
Java Developer            1820
IT Security Analyst       1714
Project Manager           1218
IT Project Manager        1119
Python Developer           689
Business Analyst           610
Full Stack Developer       512
Data Analyst               287
Software Engineer          107
Backend Developer           69
Name: count, dtype: int64

Sample profile:
                   career                                       profile_text
0  Database Administrator  skills database administration database ms sql...
1  Database Administrator  skills sql server management studio visual stu...


In [19]:
X_text = df["profile_text"].copy()
y_text = df["career"].copy()

print("X:", len(X_text))
print("Classes:", y_text.nunique())

X: 16010
Classes: 13


In [20]:
encoder = LabelEncoder()

y = encoder.fit_transform(y_text)

print("\nClasses:")
for i, cls in enumerate(encoder.classes_):
    print(i, "->", cls)


Classes:
0 -> Backend Developer
1 -> Business Analyst
2 -> Data Analyst
3 -> Database Administrator
4 -> Front End Developer
5 -> Full Stack Developer
6 -> IT Project Manager
7 -> IT Security Analyst
8 -> Java Developer
9 -> Network Administrator
10 -> Project Manager
11 -> Python Developer
12 -> Software Engineer


In [21]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTrain:", len(X_train_text))
print("Test :", len(X_test_text))


Train: 12808
Test : 3202


In [22]:
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=6000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.90,
    sublinear_tf=True,
    dtype=np.float32
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

print("TF-IDF:", X_train_tfidf.shape)

TF-IDF: (12808, 6000)


In [23]:
selector = SelectKBest(
    score_func=chi2,
    k=min(2500, X_train_tfidf.shape[1])
)

X_train = selector.fit_transform(
    X_train_tfidf,
    y_train
)

X_test = selector.transform(
    X_test_tfidf
)

print("RF features:", X_train.shape)

RF features: (12808, 2500)


In [24]:
rf_model = RandomForestClassifier(
    n_estimators=200,

    max_depth=16,

    min_samples_split=10,
    min_samples_leaf=4,

    max_features="sqrt",

    class_weight="balanced_subsample",

    bootstrap=True,

    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",16
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",4
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced_subsample'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then conside

In [58]:
# ============================================================
# NEXT RF CONFIGURATION
# ============================================================
from sklearn.metrics import f1_score
rf_model = RandomForestClassifier(
    n_estimators=250,
    max_depth=16,

    min_samples_split=12,
    min_samples_leaf=3,

    max_features=0.5,

    class_weight="balanced_subsample",

    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

train_pred = rf_model.predict(X_train)
test_pred = rf_model.predict(X_test)

train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)
balanced_acc = balanced_accuracy_score(y_test, test_pred)

print("\n==============================")
print("NEXT RANDOM FOREST")
print("==============================")

print(f"Training Accuracy : {train_acc * 100:.2f}%")
print(f"Testing Accuracy  : {test_acc * 100:.2f}%")
print(f"Balanced Accuracy : {balanced_acc * 100:.2f}%")
print(f"Overfit Gap       : {(train_acc-test_acc) * 100:.2f}%")

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        test_pred,
        target_names=encoder.classes_,
        zero_division=0
    )
)


NEXT RANDOM FOREST
Training Accuracy : 96.63%
Testing Accuracy  : 92.47%
Balanced Accuracy : 86.40%
Overfit Gap       : 4.15%

Classification Report:
                        precision    recall  f1-score   support

     Backend Developer       1.00      0.29      0.44        14
      Business Analyst       0.85      0.99      0.91       122
          Data Analyst       0.93      1.00      0.97        57
Database Administrator       1.00      0.99      0.99       506
   Front End Developer       0.98      0.98      0.98       540
  Full Stack Developer       0.91      0.98      0.94       102
    IT Project Manager       0.61      0.55      0.58       224
   IT Security Analyst       0.99      0.98      0.98       343
        Java Developer       0.99      0.98      0.99       364
 Network Administrator       0.99      0.98      0.98       527
       Project Manager       0.61      0.66      0.64       244
      Python Developer       0.99      0.99      0.99       138
     Software En

In [59]:
# RANDOM FOREST MACRO F1

rf_test_pred = rf_model.predict(X_test)

rf_macro_f1 = f1_score(
    y_test,
    rf_test_pred,
    average="macro"
)

print("=" * 40)
print("RANDOM FOREST")
print(f"Macro F1-Score: {rf_macro_f1:.4f}")
print(f"Macro F1-Score: {rf_macro_f1 * 100:.2f}%")

RANDOM FOREST
Macro F1-Score: 0.8695
Macro F1-Score: 86.95%


In [26]:
import os
import joblib

os.makedirs("models", exist_ok=True)

joblib.dump(
    rf_model,
    "models/random_forest_model.pkl"
)

joblib.dump(
    tfidf,
    "models/tfidf_vectorizer.pkl"
)

joblib.dump(
    selector,
    "models/feature_selector.pkl"
)

joblib.dump(
    encoder,
    "models/label_encoder.pkl"
)

print("Random Forest pipeline saved successfully.")

Random Forest pipeline saved successfully.


#XGBoost

In [27]:
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
import xgboost as xgb

xgb_base = xgb.XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

param_grid = {
    "n_estimators": [100, 150],
    "max_depth": [3, 4],
    "learning_rate": [0.03, 0.05],
    "min_child_weight": [5, 8],
    "subsample": [0.7, 0.8],
    "colsample_bytree": [0.7, 0.8],
    "reg_alpha": [0.5, 1.0],
    "reg_lambda": [5.0, 10.0]
}

cv = StratifiedKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)

search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,

    # Only 5 combinations
    n_iter=5,

    scoring="balanced_accuracy",

    cv=cv,

    verbose=1,

    random_state=42,

    n_jobs=-1
)

print("Starting FAST XGBoost CV...")

search.fit(
    X_train,
    y_train
)

print("\nBest Parameters:")
print(search.best_params_)

print(
    "\nBest CV Balanced Accuracy:",
    search.best_score_
)

Starting FAST XGBoost CV...
Fitting 2 folds for each of 5 candidates, totalling 10 fits

Best Parameters:
{'subsample': 0.7, 'reg_lambda': 10.0, 'reg_alpha': 1.0, 'n_estimators': 150, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.8}

Best CV Balanced Accuracy: 0.8352619830657282


In [60]:
from sklearn.metrics import f1_score

xgb_tuned = search.best_estimator_

train_pred = xgb_tuned.predict(X_train)
test_pred = xgb_tuned.predict(X_test)

train_acc = accuracy_score(
    y_train,
    train_pred
)

test_acc = accuracy_score(
    y_test,
    test_pred
)

balanced_acc = balanced_accuracy_score(
    y_test,
    test_pred
)

gap = train_acc - test_acc

print("\n==============================")
print("FAST TUNED XGBOOST")
print("==============================")

print(f"Training Accuracy : {train_acc * 100:.2f}%")
print(f"Testing Accuracy  : {test_acc * 100:.2f}%")
print(f"Balanced Accuracy : {balanced_acc * 100:.2f}%")
print(f"Overfit Gap       : {gap * 100:.2f}%")

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        test_pred,
        target_names=encoder.classes_,
        zero_division=0
    )
)


FAST TUNED XGBOOST
Training Accuracy : 94.87%
Testing Accuracy  : 93.07%
Balanced Accuracy : 85.33%
Overfit Gap       : 1.80%

Classification Report:
                        precision    recall  f1-score   support

     Backend Developer       0.67      0.14      0.24        14
      Business Analyst       0.87      1.00      0.93       122
          Data Analyst       0.93      0.96      0.95        57
Database Administrator       0.99      0.99      0.99       506
   Front End Developer       0.98      0.99      0.99       540
  Full Stack Developer       0.93      0.93      0.93       102
    IT Project Manager       0.65      0.57      0.60       224
   IT Security Analyst       0.99      0.98      0.99       343
        Java Developer       0.98      0.99      0.98       364
 Network Administrator       0.99      1.00      1.00       527
       Project Manager       0.64      0.68      0.66       244
      Python Developer       0.99      1.00      0.99       138
     Software En

In [65]:
from sklearn.metrics import f1_score

# ============================================
# XGBOOST MACRO F1
# ============================================

xgb_macro_f1 = f1_score(
    y_test,
    test_pred,
    average="macro"
)

xgb_weighted_f1 = f1_score(
    y_test,
    test_pred,
    average="weighted"
)

print("\n==============================")
print("XGBOOST MACRO F1")
print("==============================")

print(f"Macro F1-Score    : {xgb_macro_f1 * 100:.2f}%")
print(f"Weighted F1-Score : {xgb_weighted_f1 * 100:.2f}%")


XGBOOST MACRO F1
Macro F1-Score    : 85.93%
Weighted F1-Score : 92.86%


In [29]:
import sentence_transformers

print(sentence_transformers.__version__)

e:\final_infosys\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


5.7.0


In [30]:
from sentence_transformers import SentenceTransformer

SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

sbert_model = SentenceTransformer(
    SBERT_MODEL,
    device="cuda" if __import__("torch").cuda.is_available() else "cpu"
)

print("SBERT loaded successfully")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2374.05it/s]


SBERT loaded successfully


In [31]:
df["skill_text"] = (
    df["skill"]
    .fillna("")
    .astype(str)
)

df["skill_text"] = df["skill_text"].str.strip()

print(df["skill_text"].head())

0    database administration database ms sql server...
1    sql server management studio visual studio sql...
2    databases oracle 4 years oracle 10g sql linux ...
3    maintain multiple database environments redshi...
4    scrum agile software development product backl...
Name: skill_text, dtype: str


In [32]:
skill_embeddings = sbert_model.encode(
    df["skill_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("SBERT embedding shape:")
print(skill_embeddings.shape)

Batches:   0%|          | 0/251 [00:00<?, ?it/s]

Batches: 100%|██████████| 251/251 [09:29<00:00,  2.27s/it]


SBERT embedding shape:
(16010, 384)


In [33]:
import numpy as np

np.save(
    "models/skill_embeddings.npy",
    skill_embeddings
)

print("SBERT embeddings saved.")

SBERT embeddings saved.


In [34]:
career_embeddings = {}

for career in df["career"].unique():

    mask = (
        df["career"] == career
    )

    career_embeddings[career] = (
        skill_embeddings[mask].mean(axis=0)
    )

print(
    "Career embeddings:",
    len(career_embeddings)
)

Career embeddings: 13


In [35]:
np.save(
    "models/career_embeddings.npy",
    np.vstack(
        [
            career_embeddings[c]
            for c in encoder.classes_
        ]
    )
)

print("Career embeddings saved.")

Career embeddings saved.


Create career matrix

In [36]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

career_names = list(career_embeddings.keys())

career_matrix = np.vstack([
    career_embeddings[c]
    for c in career_names
])

print("Career matrix:", career_matrix.shape)
print("Careers:", career_names)

Career matrix: (13, 384)
Careers: ['Database Administrator', 'Business Analyst', 'Front End Developer', 'Data Analyst', 'Network Administrator', 'IT Security Analyst', 'Full Stack Developer', 'Project Manager', 'Software Engineer', 'Java Developer', 'Backend Developer', 'Python Developer', 'IT Project Manager']


Create the resume extraction function

In [37]:
import os
import PyPDF2
import docx

def extract_resume_text(path):

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    if path.lower().endswith(".pdf"):

        with open(path, "rb") as f:
            reader = PyPDF2.PdfReader(f)

            text = "\n".join(
                page.extract_text() or ""
                for page in reader.pages
            )

    elif path.lower().endswith(".docx"):

        document = docx.Document(path)

        text = "\n".join(
            p.text
            for p in document.paragraphs
        )

    else:
        raise ValueError(
            "Only PDF and DOCX files are supported."
        )

    return text

In [38]:
def extract_resume_info(text):

    doc = nlp(text)

    skills = []
    roles = []
    education = []

    for ent in doc.ents:

        value = ent.text.strip()

        if ent.label_ == "SKILL":

            if value.lower() not in [
                x.lower() for x in skills
            ]:
                skills.append(value)

        elif ent.label_ == "ROLE":

            if value.lower() not in [
                x.lower() for x in roles
            ]:
                roles.append(value)

        elif ent.label_ == "EDU":

            if value.lower() not in [
                x.lower() for x in education
            ]:
                education.append(value)

    return skills, roles, education

In [39]:
import numpy as np

def test_resume(resume_path, top_k=5):

    print("\n========================================")
    print("AI CAREER RECOMMENDATION")
    print("========================================")

    # ----------------------------------
    # 1. Extract resume text
    # ----------------------------------

    text = extract_resume_text(
        resume_path
    )

    print(
        f"\nResume text extracted: {len(text)} characters"
    )

    # ----------------------------------
    # 2. spaCy extraction
    # ----------------------------------

    skills, roles, education = (
        extract_resume_info(text)
    )

    print("\nEXTRACTED SKILLS:")
    print(
        ", ".join(skills)
        if skills
        else "No skills detected"
    )

    print("\nEXTRACTED ROLES:")
    print(
        ", ".join(roles)
        if roles
        else "No roles detected"
    )

    print("\nEXTRACTED EDUCATION:")
    print(
        ", ".join(education)
        if education
        else "No education detected"
    )

    # ----------------------------------
    # 3. Create model input
    # ----------------------------------

    profile_text = (
        " ".join(skills)
        + " "
        + " ".join(roles)
        + " "
        + " ".join(education)
        + " "
        + text
    )

    # ----------------------------------
    # 4. TF-IDF
    # ----------------------------------

    X = tfidf.transform(
        [profile_text]
    )

    # ----------------------------------
    # 5. Feature selection
    # ----------------------------------

    X = selector.transform(X)

    print(
        "\nModel feature shape:",
        X.shape
    )

    # ----------------------------------
    # 6. XGBoost prediction
    # ----------------------------------

    probabilities = (
        xgb_tuned.predict_proba(X)[0]
    )

    # ----------------------------------
    # 7. Top-K careers
    # ----------------------------------

    top_indices = np.argsort(
        probabilities
    )[::-1][:top_k]

    results = []

    for idx in top_indices:

        career = encoder.inverse_transform(
            [idx]
        )[0]

        confidence = (
            probabilities[idx] * 100
        )

        results.append({
            "career": career,
            "confidence": round(
                float(confidence),
                2
            )
        })

    # ----------------------------------
    # 8. Display
    # ----------------------------------

    print("\n========================================")
    print(f"TOP {top_k} JOB RECOMMENDATIONS")
    print("========================================")

    for i, result in enumerate(
        results,
        1
    ):

        print(
            f"{i}. {result['career']}"
            f"  →  "
            f"{result['confidence']:.2f}%"
        )

    return {
        "skills": skills,
        "roles": roles,
        "education": education,
        "recommendations": results
    }

In [40]:
def extract_education(text):

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    education = []

    in_education = False

    for line in lines:

        # Start education section
        if line.lower() in [
            "education",
            "educational qualification",
            "academic qualification",
            "academic background"
        ]:
            in_education = True
            continue

        # Stop at next major section
        if in_education and line.lower() in [
            "skill",
            "skills",
            "experience",
            "work experience",
            "projects",
            "project",
            "certifications",
            "certificates",
            "achievements",
            "summary",
            "objective"
        ]:
            break

        if in_education:

            # Ignore percentages
            if re.search(
                r"\b\d+(\.\d+)?\s*%",
                line
            ):
                continue

            # Ignore CGPA
            if "cgpa" in line.lower():
                continue

            # Ignore year/date lines
            if re.fullmatch(
                r"[\s\d\-–—toPresent]+",
                line,
                flags=re.IGNORECASE
            ):
                continue

            # Ignore very short noise
            if len(line.strip()) < 3:
                continue

            education.append(line.strip())

    return education

In [41]:
def extract_resume_info(text):

    doc = nlp(text)

    skills = []
    roles = []

    # ONLY spaCy for skills and roles
    for ent in doc.ents:

        value = ent.text.strip()

        if ent.label_ == "SKILL":

            if value.lower() not in [
                x.lower() for x in skills
            ]:
                skills.append(value)

        elif ent.label_ == "ROLE":

            if value.lower() not in [
                x.lower() for x in roles
            ]:
                roles.append(value)

    # Education comes directly from resume text
    education = extract_education(text)

    return skills, roles, education

In [43]:
import spacy
from spacy.pipeline import EntityRuler

# ============================================
# LOAD SPACY
# ============================================

nlp = spacy.blank("en")

ruler = nlp.add_pipe(
    "entity_ruler",
    config={"phrase_matcher_attr": "LOWER"}
)

# ============================================
# SKILL PATTERNS
# ============================================

SKILL_TERMS = [
    "python",
    "java",
    "c",
    "c++",
    "sql",
    "mysql",
    "mongodb",
    "html",
    "css",
    "javascript",
    "react",
    "react.js",
    "node.js",
    "nodejs",
    "angular",
    "flask",
    "fastapi",
    "spring",
    "spring boot",
    "hibernate",
    "docker",
    "aws",
    "azure",
    "git",
    "github",
    "machine learning",
    "deep learning",
    "tensorflow",
    "pytorch",
    "pandas",
    "numpy",
    "scikit-learn"
]

patterns = []

for skill in SKILL_TERMS:
    patterns.append({
        "label": "SKILL",
        "pattern": skill
    })

ruler.add_patterns(patterns)

print("spaCy loaded successfully")
print("Skill patterns:", len(SKILL_TERMS))

spaCy loaded successfully
Skill patterns: 32


In [44]:
print(nlp)

In [45]:
text = extract_resume_text(
    r"E:\final_infosys\final_infosys\uploads\Frontend_Developer_Resume.pdf"
)

skills, roles, education = extract_resume_info(text)

print("Skills:", skills)
print("Roles:", roles)
print("Education:", education)

Skills: ['GitHub', 'JavaScript', 'React.js', 'CSS', 'Git', 'React']
Roles: []
Education: ['[Degree, e.g. B.Tech in Computer Science]    |   [Start Year] – [End Year]', '[College / University Name]', 'CERTIFICATIONS & ACHIEVEMENTS', '• [Certification Name] – [Issuing Platform/Body], [Year]', '• [e.g. Meta Front -End Developer Certificate]', '• [Achieve ment, e.g. Built and deployed 3+ personal projects live on Vercel/Netlify]', '• [Achievement, e.g. Contributed to an open -source frontend project]', 'ADDITIONAL INFORMATION', '• Coding Profiles: LeetCode [handle] | HackerRank [handle] | GitHub [handle]', '• Languages Known:  English, [Others]', '• Willing to relocate: [Yes/No]']


In [46]:
result = test_resume(
    r"E:\final_infosys\final_infosys\uploads\Frontend_Developer_Resume.pdf",
    top_k=5
)


AI CAREER RECOMMENDATION

Resume text extracted: 2252 characters

EXTRACTED SKILLS:
GitHub, JavaScript, React.js, CSS, Git, React

EXTRACTED ROLES:
No roles detected

EXTRACTED EDUCATION:
[Degree, e.g. B.Tech in Computer Science]    |   [Start Year] – [End Year], [College / University Name], CERTIFICATIONS & ACHIEVEMENTS, • [Certification Name] – [Issuing Platform/Body], [Year], • [e.g. Meta Front -End Developer Certificate], • [Achieve ment, e.g. Built and deployed 3+ personal projects live on Vercel/Netlify], • [Achievement, e.g. Contributed to an open -source frontend project], ADDITIONAL INFORMATION, • Coding Profiles: LeetCode [handle] | HackerRank [handle] | GitHub [handle], • Languages Known:  English, [Others], • Willing to relocate: [Yes/No]

Model feature shape: (1, 2500)

TOP 5 JOB RECOMMENDATIONS
1. Front End Developer  →  98.44%
2. Backend Developer  →  0.32%
3. Project Manager  →  0.30%
4. IT Project Manager  →  0.20%
5. Network Administrator  →  0.13%


In [47]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y = encoder.fit_transform(df["career"])

In [48]:
import pandas as pd
import os

os.makedirs("models", exist_ok=True)

career_metadata = pd.DataFrame({
    "career": encoder.classes_
})

career_metadata.to_csv(
    "models/career_embeddings_meta.csv",
    index=False
)

print("✅ career_embeddings_meta.csv saved successfully.")
print(career_metadata)

✅ career_embeddings_meta.csv saved successfully.
                    career
0        Backend Developer
1         Business Analyst
2             Data Analyst
3   Database Administrator
4      Front End Developer
5     Full Stack Developer
6       IT Project Manager
7      IT Security Analyst
8           Java Developer
9    Network Administrator
10         Project Manager
11        Python Developer
12       Software Engineer


In [56]:
import pandas as pd

path = "models/career_embeddings_meta.csv"

meta = pd.read_csv(path)

# Rename career → job_title
meta = meta.rename(columns={
    "career": "job_title"
})

# Add category
meta["category"] = "Software Engineering"

# Save
meta.to_csv(path, index=False)

print("✅ Fixed metadata")
print(meta)
print(meta.columns.tolist())

✅ Fixed metadata
                 job_title              category
0        Backend Developer  Software Engineering
1         Business Analyst  Software Engineering
2             Data Analyst  Software Engineering
3   Database Administrator  Software Engineering
4      Front End Developer  Software Engineering
5     Full Stack Developer  Software Engineering
6       IT Project Manager  Software Engineering
7      IT Security Analyst  Software Engineering
8           Java Developer  Software Engineering
9    Network Administrator  Software Engineering
10         Project Manager  Software Engineering
11        Python Developer  Software Engineering
12       Software Engineer  Software Engineering
['job_title', 'category']


In [57]:
import json
import os

os.makedirs("models", exist_ok=True)

model_metrics = {
    "Random Forest": {
        "training_accuracy": float(rf_train_acc),
        "testing_accuracy": float(rf_test_acc),
        "balanced_accuracy": float(rf_balanced_acc)
    },
    "XGBoost": {
        "training_accuracy": float(xgb_train_acc),
        "testing_accuracy": float(xgb_test_acc),
        "balanced_accuracy": float(xgb_balanced_acc)
    }
}

with open("models/model_metrics.json", "w") as f:
    json.dump(model_metrics, f, indent=4)

print("✅ model_metrics.json saved successfully.")

✅ model_metrics.json saved successfully.


In [66]:
import json
import os

METRICS_PATH = r"E:\final_infosys\final_infosys\models\model_metrics.json"

with open(METRICS_PATH, "r") as f:
    metrics = json.load(f)

metrics["XGBoost"]["macro_f1"] = float(xgb_macro_f1)

with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=4)

print("\n✅ XGBoost Macro F1 saved successfully")
print(json.dumps(metrics, indent=4))


✅ XGBoost Macro F1 saved successfully
{
    "Random Forest": {
        "training_accuracy": 0.9663,
        "testing_accuracy": 0.9247,
        "balanced_accuracy": 0.864
    },
    "XGBoost": {
        "training_accuracy": 0.9487,
        "testing_accuracy": 0.9307,
        "balanced_accuracy": 0.8533,
        "macro_f1": 0.8592974568036869
    },
    "Logistic Regression": {
        "training_accuracy": 0.9584183399549018,
        "testing_accuracy": 0.9234485720420872,
        "balanced_accuracy": 0.8313167782354657,
        "macro_f1": 0.8614849333523169,
        "weighted_f1": 0.9221529881043937
    }
}
